# Resumen de Apuntes - PyTorch Workflow

Este notebook resume lo que me preguntaste y lo que te respondi sobre el flujo de trabajo de PyTorch, con foco en los puntos 2, 3, 5 y 6 de [01_pytorch_workflow.ipynb](../01_pytorch_workflow.ipynb).

## Contenido
- Punto 2: Build model
- Punto 3: Train model
- Punto 5: Saving and loading
- Punto 6: Putting it all together
- Mini ejemplos de codigo
- Seccion para integrar el enlace compartido

## Punto 2 - Build model (resumen)

Idea principal: definir una clase de modelo que herede de `nn.Module`, crear parametros entrenables y describir el calculo en `forward()`.

Claves:
- `nn.Module`: base para modelos en PyTorch.
- `nn.Parameter`: marca tensores como aprendibles (`requires_grad=True`).
- `forward(x)`: define como pasar de entrada a salida.
- En regresion lineal simple: `y_hat = w*x + b`.

Tambien se revisa:
- `model.parameters()` para ver parametros entrenables.
- `model.state_dict()` para inspeccionar estado (pesos y bias).
- Inferencia inicial con `torch.inference_mode()` para predecir sin gradientes.

In [1]:
import torch
from torch import nn

class LinearRegressionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.weights = nn.Parameter(torch.randn(1, dtype=torch.float), requires_grad=True)
        self.bias = nn.Parameter(torch.randn(1, dtype=torch.float), requires_grad=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.weights * x + self.bias

torch.manual_seed(42)
model_0 = LinearRegressionModel()
print(model_0.state_dict())

OrderedDict({'weights': tensor([0.3367]), 'bias': tensor([0.1288])})


## Punto 3 - Train model (resumen)

Idea principal: el modelo aprende ajustando parametros para minimizar una funcion de perdida.

Piezas usadas:
- Loss: `nn.L1Loss()` (MAE) para regresion.
- Optimizer: `torch.optim.SGD(...)` con `lr=0.01`.

Pasos del training loop:
1. Forward pass.
2. Calcular loss.
3. `optimizer.zero_grad()`.
4. `loss.backward()`.
5. `optimizer.step()`.

Pasos del testing loop:
- `model.eval()`
- `with torch.inference_mode():`
- Forward en test + calculo de `test_loss`.

In [2]:
# Datos de juguete para entrenar
weight, bias = 0.7, 0.3
X = torch.arange(0, 1, 0.02).unsqueeze(dim=1)
y = weight * X + bias
split = int(0.8 * len(X))
X_train, y_train = X[:split], y[:split]
X_test, y_test = X[split:], y[split:]

loss_fn = nn.L1Loss()
optimizer = torch.optim.SGD(model_0.parameters(), lr=0.01)

for epoch in range(100):
    model_0.train()
    y_pred = model_0(X_train)
    loss = loss_fn(y_pred, y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

model_0.eval()
with torch.inference_mode():
    test_pred = model_0(X_test)
    test_loss = loss_fn(test_pred, y_test)

print('train_loss:', float(loss), 'test_loss:', float(test_loss))
print(model_0.state_dict())

train_loss: 0.02479521557688713 test_loss: 0.05687814950942993
OrderedDict({'weights': tensor([0.5784]), 'bias': tensor([0.3513])})


C:\Users\egull\AppData\Local\Temp\ipykernel_18040\3934257437.py:25: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:839.)
  print('train_loss:', float(loss), 'test_loss:', float(test_loss))


## Punto 5 - Saving and loading (resumen)

Idea principal: guardar y cargar el `state_dict()` para inferencia y portabilidad.

Funciones clave:
- `torch.save(...)`
- `torch.load(...)`
- `model.load_state_dict(...)`

Recomendacion: guardar `state_dict()` en vez del modelo entero para evitar roturas por cambios de estructura del codigo.

In [3]:
from pathlib import Path

model_dir = Path('models')
model_dir.mkdir(parents=True, exist_ok=True)
save_path = model_dir / '01_pytorch_workflow_model_0_apuntes.pth'

torch.save(model_0.state_dict(), save_path)

loaded_model = LinearRegressionModel()
loaded_model.load_state_dict(torch.load(save_path))
loaded_model.eval()

with torch.inference_mode():
    loaded_preds = loaded_model(X_test)

print('Mismo shape de predicciones:', loaded_preds.shape == test_pred.shape)

Mismo shape de predicciones: True


## Punto 6 - Putting it all together (resumen)

Idea principal: repetir todo el pipeline de forma limpia y device-agnostic (CPU/GPU).

Flujo integrado:
1. Detectar dispositivo: `cuda` si disponible, si no `cpu`.
2. Crear datos y split train/test.
3. Definir modelo (muchas veces con `nn.Linear`).
4. Definir loss y optimizer.
5. Entrenar y evaluar por epochs.
6. Confirmar parametros aprendidos y hacer inferencia final.

Practica clave: modelo y datos deben estar en el mismo device para evitar errores de runtime.

In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

class LinearRegressionModelV2(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)

    def forward(self, x):
        return self.linear(x)

model_1 = LinearRegressionModelV2().to(device)
X_train_d, y_train_d = X_train.to(device), y_train.to(device)
X_test_d, y_test_d = X_test.to(device), y_test.to(device)

loss_fn = nn.L1Loss()
optimizer = torch.optim.SGD(model_1.parameters(), lr=0.01)

for epoch in range(200):
    model_1.train()
    pred = model_1(X_train_d)
    loss = loss_fn(pred, y_train_d)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

model_1.eval()
with torch.inference_mode():
    pred_test = model_1(X_test_d)
    print('test_loss_v2:', float(loss_fn(pred_test, y_test_d)))

Using device: cuda
test_loss_v2: 0.13695549964904785


## Aportes del enlace compartido

Enlace recibido: https://chatgpt.com/share/6a282a3c-41d8-83eb-90e1-f8ce2832cbe5

Estado actual:
- El contenido no fue accesible automaticamente desde aqui (la pagina devolvio vista de login/cookies).
- Esta seccion esta preparada para integrar literalmente el texto que pegues del enlace.

Cuando lo compartas en el chat, lo incorporo en una nueva version del notebook (sin perder lo ya implementado).

## Resumen ultra corto

- Punto 2: construir clase de modelo con `nn.Module`.
- Punto 3: entrenar con loss + optimizer + loop.
- Punto 5: guardar/cargar via `state_dict()`.
- Punto 6: integrar todo con soporte CPU/GPU.

Este notebook es tu base de apuntes editable.